In [ ]:
!pip install openml

In [ ]:
from google.colab import userdata
from huggingface_hub import login
llama_token = userdata.get('llama_token')
login(llama_token)

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm
import openml


In [ ]:
# Настройка промпта для Car

prompt_config = {
    "task": "Predict car evaluation (unacceptable, acceptable, good, very good)",
    "labels": ["unacceptable", "acceptable", "good", "very good"],
    "entity": "Car",
    "question": "What is the evaluation of this car?"
}

openml_id = 40975

output_dir = "/content/drive/MyDrive/finetuned_llama_lora_with_Car"

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464, prompt_config=None):
    dataset = openml.datasets.get_dataset(openml_id)
    target_col = dataset.default_target_attribute
    X, y, _, _ = dataset.get_data(dataset_format="dataframe", target=target_col)

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])
        class_names = prompt_config['labels']
    else:
        class_names = sorted(df[target_name].unique().tolist())

    return df, feature_names, target_name, class_names


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):

    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name, class_names = load_dataset(openml_id, prompt_config)
train_df, val_df, test_df = split_dataset(df, target_name)

df.head(5)

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,2
1,vhigh,vhigh,2,2,small,med,2
2,vhigh,vhigh,2,2,small,high,2
3,vhigh,vhigh,2,2,med,low,2
4,vhigh,vhigh,2,2,med,med,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   buying    1728 non-null   category
 1   maint     1728 non-null   category
 2   doors     1728 non-null   category
 3   persons   1728 non-null   category
 4   lug_boot  1728 non-null   category
 5   safety    1728 non-null   category
 6   class     1728 non-null   int64   
dtypes: category(6), int64(1)
memory usage: 24.7 KB


In [ ]:
df['class'].value_counts()

,count
class,
2,1210
0,384
1,69
3,65


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['labels'][int(row[target_name])]
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

The category of buying is vhigh. The category of maint is med. The category of doors is 3. The category of persons is 2. The category of lug_boot is small. The category of safety is med.: class -> good
The category of buying is vhigh. The category of maint is med. The category of doors is 3. The category of persons is 2. The category of lug_boot is small. The category of safety is med.


,buying,maint,doors,persons,lug_boot,safety,class
244,vhigh,med,3,2,small,med,2


In [ ]:
def parse_prediction(response, prompt_config):
    """Парсинг ответа модели в номер класса"""
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    # Проверка каждого класса
    for i, class_name in enumerate(prompt_config['labels']):
        class_lower = class_name.lower()

        # Точное совпадение
        if response == class_lower:
            return i

        # Начинается с имени класса
        if response.startswith(class_lower):
            return i

        # Содержит как отдельное слово
        if class_lower in response.split():
            return i

    # Не удалось распознать - возвращаем первый класс
    print(f"Warning: Could not parse '{response}' (expected one of {prompt_config['labels']})")
    return 0

response = "good"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

response = "no"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'good'
Prediction: 2
Response: 'no'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    pr = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

    if y_prob is not None:
        try:
            roc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except:
            roc = 0.0
    else:
        roc = 0.0
    return roc, f1, acc, pr, rec

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    """Bootstrap метрики с доверительными интервалами"""
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            acc = accuracy_score(y_true_boot, y_pred_boot)
            f1 = f1_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)

            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
            else:
                auc = 0.0

            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Название модели: Llama 3.1 8B Instruct от Meta
model_name = "meta-llama/Llama-3.1-8B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

# Загрузка токенизатора для Llama 3.1
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / 1e9
    print(f"Занято видеопамяти на GPU: {allocated_memory:.2f} GB")

Используемое устройство: cuda


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Занято видеопамяти на GPU: 16.06 GB


In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"Answer with only one word from: '{labels_str}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['labels'][int(ex[target_name])]

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для всех классов
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)

    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    # Возвращаем ответ и вероятности всех классов
    return response, probs.cpu().numpy()

# Тест
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, probs = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}'")
print(f"Probabilities: {dict(zip(prompt_config['labels'], probs))}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'acceptable'
Probabilities: {'unacceptable': np.float32(0.119833246), 'acceptable': np.float32(0.6895928), 'good': np.float32(0.17435636), 'very good': np.float32(0.016217668)}


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
# Проверка баланса классов
seed=42
labels_counts = train_df[target_name].value_counts()
print(f"\nДо балансировки:")
for cls, count in labels_counts.items():
    print(f"  Класс {prompt_config['labels'][cls]}: {count}")

# Oversample до максимального класса
max_count = labels_counts.max()
balanced_dfs = []

for cls in labels_counts.index:
    df_labels = train_df[train_df[target_name] == cls]
    if len(df_labels) < max_count:
        df_upsampled = resample(
            df_labels,
            replace=True,
            n_samples=max_count,
            random_state=seed
        )
        balanced_dfs.append(df_upsampled)
    else:
        balanced_dfs.append(df_labels)

train_df_balanced = pd.concat(balanced_dfs)
train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nПосле балансировки:")
for cls, count in train_df_balanced[target_name].value_counts().items():
    print(f"  Класс {prompt_config['labels'][cls]}: {count}")



До балансировки:
  Класс good: 726
  Класс unacceptable: 230
  Класс acceptable: 41
  Класс very good: 39

После балансировки:
  Класс good: 726
  Класс unacceptable: 726
  Класс acceptable: 726
  Класс very good: 726


In [ ]:
def create_dataset(df, desc="Подготовка данных"):
    """Создание dataset для обучения"""
    labels_str = "', '".join(prompt_config['labels'])
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}. "
        f"Answer with only one word from: '{labels_str}'."
    )
    texts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        input_text = row_to_text_template(row, feature_names, target_name)
        target = prompt_config['labels'][int(row[target_name])]

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts

train_texts = create_dataset(train_df_balanced)
train_dataset = Dataset.from_dict({"text": train_texts})

Подготовка данных:   0%|          | 0/2904 [00:00<?, ?it/s]

In [ ]:
# Токенизация
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(f"\n Подготовлено {len(tokenized_dataset)} примеров для обучения")

Map:   0%|          | 0/2904 [00:00<?, ? examples/s]


 Подготовлено 2904 примеров для обучения


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

num_epochs = 10
batch_size = 16

# Загрузка базовой модели
model_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    dtype=torch.bfloat16,
)

# Настройка LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

# Аргументы для обучения
training_args_lora = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch_fused",
    warmup_steps=50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    report_to="none",
    dataloader_num_workers=4,
    group_by_length=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Mounted at /content/drive


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Обучение
print(f"\nНачинаем обучение на {num_epochs} эпох")

train_start_time = time.time()
trainer_lora.train()
training_time = time.time() - train_start_time

print(f"Обучение завершено за {training_time/60:.1f} минут")

trainer_lora.save_model(output_dir)
print(f"Модель сохранена в {output_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Начинаем обучение на 10 эпох


Step,Training Loss
10,3.039808
20,1.242680
30,0.221990
40,0.110951
50,0.093598
60,0.080373
70,0.084203
80,0.079119
90,0.076945
100,0.075980


Обучение завершено за 11.2 минут
Модель сохранена в /content/drive/MyDrive/finetuned_llama_lora_with_Car


In [ ]:
import glob
import re
from peft import PeftModel
print("\nОценка checkpoint на val")

# Поиск и сортировка checkpoint
checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
checkpoints_sorted = sorted(
    checkpoints,
    key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
)

print(f"\nНайдено {len(checkpoints_sorted)} checkpoints\n")

# Хранение результатов
results_list = []
best_auc = 0.0
best_checkpoint = None

# Оценка каждого checkpoint
for checkpoint_path in checkpoints_sorted:
    checkpoint_name = checkpoint_path.split('/')[-1]
    checkpoint_num = int(re.search(r'checkpoint-(\d+)', checkpoint_name).group(1))

    # Загрузка модели
    model_eval = PeftModel.from_pretrained(base_model, checkpoint_path)
    model_eval.eval()

    # Оценка
    y_true, y_pred, y_prob = [], [], []
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=checkpoint_name):
        prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
        response, probs = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer, device)
        prediction = parse_prediction(response, prompt_config)

        y_true.append(row[target_name])
        y_pred.append(prediction)
        y_prob.append(probs)

    # Метрики
    roc, f1, acc, pr, rec = compute_metrics(y_true, y_pred, y_prob)

    results_list.append({
        'Checkpoint': checkpoint_num,
        'ROC-AUC': roc,
        'F1': f1,
        'Accuracy': acc,
        'Precision': pr,
        'Recall': rec
    })

    if roc > best_auc:
        best_auc = roc
        best_checkpoint = checkpoint_path

    del model_eval
    torch.cuda.empty_cache()


Оценка checkpoint на val

Найдено 10 checkpoints



checkpoint-91:   0%|          | 0/346 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-182:   0%|          | 0/346 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


...

checkpoint-728:   0%|          | 0/346 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-819:   0%|          | 0/346 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-910:   0%|          | 0/346 [00:00<?, ?it/s]

In [ ]:
print(f"\nЛучший checkpoint: {best_checkpoint}")


Лучший checkpoint: /content/drive/MyDrive/finetuned_llama_lora_with_Car/checkpoint-728


In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('Checkpoint')
results_df

,Checkpoint,ROC-AUC,F1,Accuracy,Precision,Recall
0,91,0.682048,0.057827,0.098266,0.127797,0.253837
1,182,0.922629,0.499031,0.763006,0.469524,0.573347
2,273,0.986766,0.601253,0.852601,0.621821,0.618654
3,364,0.994987,0.653197,0.913295,0.623230,0.695691
4,455,0.997844,0.716929,0.924855,0.701565,0.734357
5,546,0.999448,0.732647,0.947977,0.732692,0.733766
6,637,0.999822,0.739219,0.956647,0.730825,0.747934
7,728,1.000000,0.743461,0.962428,0.737255,0.750000
8,819,1.000000,0.743461,0.962428,0.737255,0.750000
9,910,0.999956,0.739219,0.956647,0.730825,0.747934


In [ ]:
model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint)
model_finetuned.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# Оценка на тестовой выборке
y_true_fine, y_pred_fine, y_prob_fine = [], [], []

start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test evaluation"):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    response, probs = predict_single_with_prob(prompt, prompt_config, model_finetuned, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_fine.append(row[target_name])
    y_pred_fine.append(prediction)
    y_prob_fine.append(probs)

test_time = time.time() - start_time

print(f"\nВремя оценки: {test_time:.1f}s ({test_time / len(y_true_fine):.3f}s/sample)")


Test evaluation:   0%|          | 0/346 [00:00<?, ?it/s]


Время оценки: 20.6s (0.060s/sample)


In [ ]:
# Вычисляем метрики

roc_fine, f1_fine, acc_fine, pr_fine, rec_fine = compute_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine)
)
print("Результаты fine tune:")
print(f"ROC-AUC: {roc_fine}")
print(f"F1 Score: {f1_fine}")
print(f"Accuracy: {acc_fine}")
print(f"Precision: {pr_fine}")
print(f"Recall: {rec_fine}")


Результаты fine tune:
ROC-AUC: 0.9999155119972963
F1 Score: 0.7370967741935484
Accuracy: 0.953757225433526
Precision: 0.7298102160306885
Recall: 0.7446871310507674


In [ ]:
fine_tuned_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine),
    n_iter=1000
)

print("\nРезультаты fine-tune (bootstrap метрики с доверительными интервалами):")
for key, value in fine_tuned_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты fine-tune (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.9999±0.0001
  F1: 0.7368±0.0041
  Accuracy: 0.9537±0.0111
  Precision: 0.7296±0.0057
  Recall: 0.7444±0.0037


In [ ]:
from google.colab import runtime
runtime.unassign()

Время обучения: 11 мин

Время выборки моделя: ~ 5 мин

Время тестирования: ~ 20 сек

Использовано оперативная памяти графического процесса: 53.2/95.6GB.

Графический процессор G4 95.6GB.